In [38]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class ImprovedNeuralNetwork:
    def __init__(self, input_size=2, hidden_size=4, learning_rate=0.01):
        # Initialisation plus stable avec Xavier/Glorot
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate

        # Hyperparamètres ajustés
        self.dropout_rate = 0.2
        self.lambda_reg = 0.01
        self.beta = 0.9

        # Initialisation des poids
        self._initialize_parameters()

        # Historiques pour monitoring
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.learning_rates = []

    def _initialize_parameters(self):
        # Xavier/Glorot initialization
        self.w1 = np.random.randn(self.input_size, self.hidden_size) * np.sqrt(2.0/(self.input_size + self.hidden_size))
        self.b1 = np.zeros((1, self.hidden_size))
        self.w2 = np.random.randn(self.hidden_size, 1) * np.sqrt(2.0/(self.hidden_size + 1))
        self.b2 = np.zeros((1, 1))

        # Momentum
        self.vw1 = np.zeros_like(self.w1)
        self.vw2 = np.zeros_like(self.w2)
        self.vb1 = np.zeros_like(self.b1)
        self.vb2 = np.zeros_like(self.b2)

    def _get_learning_rate(self, epoch, total_epochs):
        if epoch < total_epochs * 0.2:
            return self.learning_rate
        elif epoch < total_epochs * 0.4:
            return self.learning_rate * 0.1
        elif epoch < total_epochs * 0.6:
            return self.learning_rate * 0.01
        else:
            return self.learning_rate * 0.001

    def forward(self, x, training=True):
        # Forward pass avec dropout
        self.z1 = np.dot(x, self.w1) + self.b1
        self.a1 = np.tanh(self.z1)

        if training:
            self.dropout_mask = np.random.binomial(1, 1-self.dropout_rate, self.a1.shape) / (1-self.dropout_rate)
            self.a1 *= self.dropout_mask

        self.z2 = np.dot(self.a1, self.w2) + self.b2
        self.a2 = 1 / (1 + np.exp(-self.z2))
        return self.a2

    def compute_cost(self, y_true, y_pred):
        m = y_true.shape[0]
        # Cross-entropy avec régularisation L2
        cross_entropy = -np.mean(y_true * np.log(y_pred + 1e-15) + (1-y_true) * np.log(1 - y_pred + 1e-15))
        l2_reg = (self.lambda_reg/2) * (np.sum(np.square(self.w1)) + np.sum(np.square(self.w2)))
        return cross_entropy + l2_reg

    def backward(self, x, y, output, current_lr):
        m = x.shape[0]

        # Gradients avec régularisation L2
        dz2 = output - y
        dw2 = (1/m) * np.dot(self.a1.T, dz2) + self.lambda_reg * self.w2
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)

        da1 = np.dot(dz2, self.w2.T)
        da1 *= self.dropout_mask  # Appliquer le masque dropout
        dz1 = da1 * (1 - np.power(self.a1, 2))
        dw1 = (1/m) * np.dot(x.T, dz1) + self.lambda_reg * self.w1
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)

        # Momentum update
        self.vw2 = self.beta * self.vw2 + (1 - self.beta) * dw2
        self.vb2 = self.beta * self.vb2 + (1 - self.beta) * db2
        self.vw1 = self.beta * self.vw1 + (1 - self.beta) * dw1
        self.vb1 = self.beta * self.vb1 + (1 - self.beta) * db1

        # Gradient clipping
        clip_value = 1.0
        for grad in [self.vw1, self.vw2, self.vb1, self.vb2]:
            np.clip(grad, -clip_value, clip_value, out=grad)

        # Mise à jour des paramètres
        self.w2 -= current_lr * self.vw2
        self.b2 -= current_lr * self.vb2
        self.w1 -= current_lr * self.vw1
        self.b1 -= current_lr * self.vb1

    def train(self, x_raw, y, epochs=1000, batch_size=2, validation_split=0.2):
        noise_scale = 0.1
        x_train = x_train + np.random.normal(0, noise_scale, x_train.shape)

        # Split train/validation
        x_train, x_val, y_train, y_val = train_test_split(x_raw, y, test_size=validation_split, random_state=42)

        # Normalisation
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_val = scaler.transform(x_val)

        n_samples = x_train.shape[0]
        best_val_loss = float('inf')
        patience = 20
        wait = 0

        for epoch in range(epochs):
            current_lr = self._get_learning_rate(epoch, epochs)
            self.learning_rates.append(current_lr)

            # Training
            indices = np.random.permutation(n_samples)
            total_train_loss = 0
            total_train_acc = 0

            for i in range(0, n_samples, batch_size):
                batch_indices = indices[i:min(i + batch_size, n_samples)]
                x_batch = x_train[batch_indices]
                y_batch = y_train[batch_indices]

                output = self.forward(x_batch, training=True)
                self.backward(x_batch, y_batch, output, current_lr)

                batch_loss = self.compute_cost(y_batch, output)
                batch_acc = np.mean((output >= 0.5) == y_batch)

                total_train_loss += batch_loss * len(batch_indices)
                total_train_acc += batch_acc * len(batch_indices)

            # Validation
            val_output = self.forward(x_val, training=False)
            val_loss = self.compute_cost(y_val, val_output)
            val_acc = np.mean((val_output >= 0.5) == y_val)

            # Métriques moyennes
            train_loss = total_train_loss / n_samples
            train_acc = total_train_acc / n_samples

            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}")
                print(f"Train - Loss: {train_loss:.6f}, Acc: {train_acc:.4f}")
                print(f"Val - Loss: {val_loss:.6f}, Acc: {val_acc:.4f}")
                print(f"Learning Rate: {current_lr:.6f}\n")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

        # Visualisation
        self._plot_training_history()

        return self.train_losses, self.val_losses

    def _plot_training_history(self):
        plt.figure(figsize=(15, 5))

        # Loss
        plt.subplot(1, 3, 1)
        plt.plot(self.train_losses, label='Train')
        plt.plot(self.val_losses, label='Validation')
        plt.title('Évolution de la Loss')
        plt.xlabel('Époques')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        # Accuracy
        plt.subplot(1, 3, 2)
        plt.plot(self.train_accuracies, label='Train')
        plt.plot(self.val_accuracies, label='Validation')
        plt.title('Évolution de l\'Accuracy')
        plt.xlabel('Époques')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        # Learning Rate
        plt.subplot(1, 3, 3)
        plt.plot(self.learning_rates)
        plt.title('Évolution du Learning Rate')
        plt.xlabel('Époques')
        plt.ylabel('Learning Rate')
        plt.grid(True)

        plt.tight_layout()
        plt.show()

# Test du modèle
X = np.array([[1, 8], [2, 7], [3, 6], [4, 6], [5, 5], [6, 5], [7, 4], [8, 3]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1]).reshape(-1, 1)

model = ImprovedNeuralNetwork(input_size=2, hidden_size=4, learning_rate=0.01)
history = model.train(X, y, epochs=1000, batch_size=2)

UnboundLocalError: cannot access local variable 'x_train' where it is not associated with a value